# Fig 7 - Model-human alignment
Pearson and Spearman correlations between each model and the human ratings, on both the explicit (discrete) response and the implicit (token-probability) expected score.

## Load data

In [ ]:
"""Load the merged data using the project's data_loading module."""
import sys
sys.path.insert(0, r"C:\\Users\\PauloPC2019\\Desktop\\Analysis scriptie")
from data_loading import load_data

df = load_data()
df.head()


## Compute the six correlations

In [ ]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr

models = [
    ("Qwen 2.5-72B Instruct",  "qwen"),
    ("Llama 3.3-70B Instruct", "llama_instruct"),
    ("Llama 3.1-70B Base",     "llama_base"),
]

rows = []
for name, prefix in models:
    for measure_label, col in [("Explicit (response)", f"{prefix}_response"),
                                ("Implicit (expected)", f"{prefix}_expected")]:
        sub = df[["human", col]].dropna()
        r,   rp = pearsonr(sub["human"], sub[col])
        rho, sp = spearmanr(sub["human"], sub[col])
        rows.append({
            "Model": name,
            "Measure": measure_label,
            "N": len(sub),
            "Pearson r": r,
            "Spearman rho": rho,
        })

res = pd.DataFrame(rows)
res


## Build the figure (paired bars: Pearson | Spearman, Explicit vs. Implicit)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

models   = ["Qwen 2.5-72B Instruct", "Llama 3.3-70B Instruct", "Llama 3.1-70B Base"]
measures = ["Explicit (response)", "Implicit (expected)"]
colors   = {"Explicit (response)": "#7c7c7c", "Implicit (expected)": "#1f77b4"}

pearson  = np.zeros((len(models), len(measures)))
spearman = np.zeros((len(models), len(measures)))
for i, m in enumerate(models):
    for j, ms in enumerate(measures):
        row = res[(res["Model"] == m) & (res["Measure"] == ms)].iloc[0]
        pearson[i, j]  = row["Pearson r"]
        spearman[i, j] = row["Spearman rho"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), dpi=150, sharey=True)
x = np.arange(len(models))
w = 0.35
for ax, mat, title in zip(axes, [pearson, spearman],
                          ["Pearson correlation with human ratings",
                           "Spearman correlation with human ratings"]):
    for j, ms in enumerate(measures):
        bars = ax.bar(x + (j - 0.5) * w, mat[:, j], w,
                      label=ms, color=colors[ms], edgecolor="black", linewidth=0.6)
        for b, v in zip(bars, mat[:, j]):
            ax.text(b.get_x() + b.get_width()/2, v + 0.012, f"{v:.2f}",
                    ha="center", va="bottom", fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace(" ", "\n", 1) for m in models], fontsize=10)
    ax.set_ylim(0, 0.75)
    ax.set_ylabel("Correlation with human ratings", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.grid(True, axis="y", linestyle="--", alpha=0.4)
    ax.legend(fontsize=10, loc="upper right")

fig.suptitle("Model-human alignment: explicit responses vs. implicit (token-probability) expected scores",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig7_human_alignment.png", dpi=200, bbox_inches="tight")
plt.show()
